# Ayudantía 14

### Cargar datos

Realizamos esto con el códifo visto en la Ayudantía 13

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

vehiculos = pd.read_csv('vehicles.csv')


In [ ]:
nulos = vehiculos.isnull().sum()
print(nulos)

year               0
make               0
model             12
trim             151
body            1309
transmission    8514
vin                0
state              0
condition       5288
odometer          28
color            114
interior         114
seller             0
mmr                0
sellingprice       0
saledate           0
dtype: int64


In [35]:
vehiculos = vehiculos.dropna()

In [36]:
nulos = vehiculos.isnull().sum()
print(nulos)

year            0
make            0
model           0
trim            0
body            0
transmission    0
vin             0
state           0
condition       0
odometer        0
color           0
interior        0
seller          0
mmr             0
sellingprice    0
saledate        0
dtype: int64


In [48]:
import sqlite3

conn = sqlite3.connect('vehicles.db')

cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS Manufacturers")
cursor.execute("DROP TABLE IF EXISTS Models")
cursor.execute("DROP TABLE IF EXISTS States")
cursor.execute("DROP TABLE IF EXISTS Sellers")
cursor.execute("DROP TABLE IF EXISTS Vehicles")
cursor.execute("DROP TABLE IF EXISTS Sales")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Manufacturers (
    manufacturer_id INTEGER PRIMARY KEY,
    make TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Models (
    model_id INTEGER PRIMARY KEY,
    model_name TEXT NOT NULL,
    trim TEXT,
    manufacturer_id INTEGER,
    FOREIGN KEY (manufacturer_id) REFERENCES Manufacturers (manufacturer_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS States (
    state_id INTEGER PRIMARY KEY,
    state TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Sellers (
    seller_id INTEGER PRIMARY KEY,
    seller_name TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Vehicles (
    vin TEXT PRIMARY KEY,
    year INTEGER,
    body TEXT,
    transmission TEXT,
    condition INTEGER,
    odometer INTEGER,
    color TEXT,
    interior TEXT,
    model_id INTEGER,
    state_id INTEGER,
    seller_id INTEGER,
    FOREIGN KEY (model_id) REFERENCES Models (model_id),
    FOREIGN KEY (state_id) REFERENCES States (state_id),
    FOREIGN KEY (seller_id) REFERENCES Sellers (seller_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Sales (
    sale_id INTEGER PRIMARY KEY,
    vin TEXT,
    mmr REAL,
    sellingprice REAL,
    saledate DATE,
    FOREIGN KEY (vin) REFERENCES Vehicles (vin)
);
""")


In [49]:
vehiculos = vehiculos.dropna(subset=['make'])

for make in vehiculos['make'].unique():
    cursor.execute("INSERT INTO Manufacturers (make) VALUES (?)", (make,))

conn.commit()

In [50]:
models = vehiculos.dropna(subset=['model', 'make'])

for _, row in models.iterrows():
    cursor.execute("SELECT manufacturer_id FROM Manufacturers WHERE make = ?", (row['make'],))
    manufacturer_id = cursor.fetchone()

    if manufacturer_id:
        manufacturer_id = manufacturer_id[0]  

        cursor.execute("""
        INSERT INTO Models (model_name, trim, manufacturer_id) 
        VALUES (?, ?, ?)
        """, (row['model'], row.get('trim', None), manufacturer_id))

conn.commit()

In [51]:
vehiculos = vehiculos.dropna(subset=['state'])

for state in vehiculos['state'].unique():
    cursor.execute("INSERT OR IGNORE INTO States (state) VALUES (?)", (state,))

conn.commit()

In [52]:
vehiculos = vehiculos.dropna(subset=['seller'])

for seller_name in vehiculos['seller'].unique():
    cursor.execute("INSERT OR IGNORE INTO Sellers (seller_name) VALUES (?)", (seller_name,))
conn.commit()


In [53]:
for _, row in vehiculos.iterrows():
    cursor.execute("SELECT model_id FROM Models WHERE model_name = ?", (row['model'],))
    model_id = cursor.fetchone()

    cursor.execute("SELECT state_id FROM States WHERE state = ?", (row['state'],))
    state_id = cursor.fetchone()

    cursor.execute("SELECT seller_id FROM Sellers WHERE seller_name = ?", (row['seller'],))
    seller_id = cursor.fetchone()

    if model_id and state_id and seller_id:
        model_id = model_id[0] 
        state_id = state_id[0]  
        seller_id = seller_id[0]  

        cursor.execute("""
        INSERT OR IGNORE INTO Vehicles (vin, year, body, transmission, condition, odometer, color, interior, model_id, state_id, seller_id) 
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (row['vin'], row['year'], row['body'], row['transmission'], row['condition'], row['odometer'], row['color'], row['interior'], model_id, state_id, seller_id))

conn.commit()

In [54]:
ventas = vehiculos.dropna(subset=['vin'])

for _, row in ventas.iterrows():
    cursor.execute("SELECT vin FROM Vehicles WHERE vin = ?", (row['vin'],))
    vin = cursor.fetchone()

    if vin:
        vin = vin[0]

        cursor.execute("""
        INSERT INTO Sales (vin, mmr, sellingprice, saledate)
        VALUES (?, ?, ?, ?)
        """, (vin, row['mmr'], row['sellingprice'], row['saledate']))

conn.commit()

### Crear una función para obtener resultados de consultas

In [55]:
def ejecutar(consulta):
    cursor.execute(consulta)
    resultado = cursor.fetchall()
    return resultado

## 1. Liste todos los fabricantes con la cantidad de vehículos asociados

In [56]:
consulta = '''
SELECT Mf.make as fabricante, COUNT(V.vin) as cantidad
FROM Manufacturers Mf
JOIN Models Md ON Mf.manufacturer_id = Md.manufacturer_id
JOIN Vehicles V ON Md.model_id = V.model_id
GROUP BY Mf.manufacturer_id
ORDER BY COUNT(V.vin) DESC
'''

resultado = ejecutar(consulta)

print('   Fabricante  | Cantidad de vehículos')
print('--------------------------------------')
for fabricante, cantidad in resultado:
    print(f'{fabricante:14} | {cantidad:20}')

   Fabricante  | Cantidad de vehículos
--------------------------------------
Ford           |                 9645
Chevrolet      |                 6326
Nissan         |                 4978
Toyota         |                 4102
Honda          |                 3404
Dodge          |                 3163
Hyundai        |                 2284
BMW            |                 2276
Kia            |                 1919
Chrysler       |                 1880
Mercedes-Benz  |                 1745
Infiniti       |                 1691
Jeep           |                 1508
Volkswagen     |                 1293
Lexus          |                 1265
GMC            |                 1016
Mazda          |                  885
Cadillac       |                  807
Subaru         |                  695
Acura          |                  644
Audi           |                  585
Lincoln        |                  577
Buick          |                  521
Pontiac        |                  519
Mitsubishi

## 2. Encuentre todos los vehículos registrados en un estado específico

In [57]:
estado = 'ny'

consulta = f'''
SELECT V.vin as vin
FROM Vehicles V
JOIN Models Md ON V.model_id = Md.model_id
JOIN States S ON V.state_id = S.state_id
WHERE S.state = '{estado}'
'''

resultado = ejecutar(consulta)

print(f'Vehículos registrados en {estado}')
print('---------------------------')
for vin in resultado:
    print(vin[0])

Vehículos registrados en ny
---------------------------
2gnflgek8e6170066
1fmjk2a51eef47613
1fadp3e21el274277
1c3cdfba6dd100956
salsk25468a141213
1c4njrbb2dd274997
1gccs149598108845
1fmhk8d8xcga46275
2lmhj5ar3bbj54925
3fa6p0h91dr160490
1d4hb48d25f557639
1fahp2f85eg186145
2hgfa1f52ah326459
1fm5k8f82dgb80800
1g8aj55f36z100927
1fmcu9j91dub05448
1ftfx1efxbfc09092
2fmdk4gc5dbb19519
2gcek19jx81110622
2fmdk4kc3dbb40912
1fmdk06166ga48481
1ftfw1etxcfb67808
1fmjk2a55bef40238
4a32b3ff3ce024099
3fahp0hg7br264003
jtebu11f870095970
1gyek63n94r115623
1gcjk33d46f152478
2a8hr54p58r830433
1fbss3bl4eda89880
4s3bmbd68e3007856
1ftww31r58ea21755
jn8az18w99w111931
1gkfk63868j211254
1fmcu9eg5ckb37936
wba3b5c53df592392
1g1jc6sg1d4173154
1fadp3f23el314406
kndpcca63b7168834
jm1bl1l8xc1530028
2fmdk4jc0dbb48922
1ftfx1ef4bfc34103
kmhct4ae9du426849
2fmdk4kc9eba75128
2fmdk4jc3dba88683
1fmju2a55eef47687
2fmhk6cc9cbd11186
1d7hu16d44j154854
1gczgfba7a1129652
1ftfw1ef7efb62658
2lmhj5at3cbl52448
5j6rm4h78el047121
2t3bf4dv

## 3. Calcule el precio promedio de venta por tipo de carrocería.

In [61]:
# Calcule el precio promedio de venta por tipo de carrocer´ ıa.
consulta = '''
SELECT V.body as carroceria, ROUND(AVG(S.sellingprice), 2) as precio_promedio
FROM Vehicles V
JOIN Sales S ON V.vin = S.vin
GROUP BY V.body
'''

resultado = ejecutar(consulta)

print('     Carrocería    | Precio promedio')
print('------------------------------------')
for carroceria, precio_promedio in resultado:
    print(f'{carroceria:18} | {precio_promedio:15}')

     Carrocería    | Precio promedio
------------------------------------
Access Cab         |        12269.35
Beetle Convertible |        17166.67
CTS Coupe          |        22671.88
CTS Wagon          |        21833.33
CTS-V Coupe        |        35283.33
CTS-V Wagon        |         50500.0
Cab Plus           |          4400.0
Cab Plus 4         |          7900.0
Club Cab           |         4471.43
Convertible        |         16225.7
Coupe              |        14172.39
Crew Cab           |        20728.81
CrewMax Cab        |        25609.26
Double Cab         |        22657.33
E-Series Van       |        18737.87
Elantra Coupe      |        12663.64
Extended Cab       |         9974.62
G Convertible      |        24931.63
G Coupe            |        22787.22
G Sedan            |         20372.6
G37 Convertible    |         19712.5
G37 Coupe          |         16100.0
Genesis Coupe      |        14722.22
Hatchback          |        10240.02
King Cab           |         7554.24
K

## 4. Liste los vehículos con la mayor diferencia entre el precio de venta y el valor de mercado (MMR) ordenados de mayor a menor

In [ ]:
consulta = '''
SELECT V.vin as vin, S.sellingprice as precio, S.mmr as valor, S.sellingprice - S.mmr as diferencia
FROM Sales S
JOIN Vehicles V ON S.vin = V.vin
ORDER BY diferencia DESC
LIMIT 10
'''

resultado = ejecutar(consulta)

print('       VIN        |  Precio  |   Valor  | Diferencia')
print('----------------------------------------------------')
for vin, precio, valor, diferencia in resultado:
    print(f'{vin:12} | {precio:8} | {valor:8} | {diferencia:9}')

       VIN        |  Precio  |   Valor  | Diferencia
----------------------------------------------------
1gnskjkc4fr158207 |  46400.0 |  11100.0 |   35300.0
4f2yz04136km20229 |  35000.0 |   8500.0 |   26500.0
wddug8cb8ea054954 | 106400.0 |  83000.0 |   23400.0
3gcukrec8eg233825 |  36250.0 |  14200.0 |   22050.0
3fadp4bj5em142820 |  30900.0 |   9850.0 |   21050.0
2gnalcek7e6105115 |  38000.0 |  17650.0 |   20350.0
1n4al3ap9ec119301 |  37200.0 |  16900.0 |   20300.0
3fa6p0h73dr107451 |  34500.0 |  14700.0 |   19800.0
1gnskje71er203407 |  37200.0 |  17500.0 |   19700.0
1fahp2e83eg145921 |  38000.0 |  19300.0 |   18700.0


## 5. Identifique los estados con el mayor número de ventas y el precio promedio más alto

In [75]:
consulta = '''
SELECT St.state as estado, COUNT(S.sale_id) as cantidad
FROM Sales S
JOIN Vehicles V ON S.vin = V.vin
JOIN States St ON V.state_id = St.state_id
GROUP BY St.state_id
ORDER BY cantidad DESC
LIMIT 5
'''

resultado = ejecutar(consulta)

print(' Estado | Cantidad de ventas')
print('----------------------------')
for estado, cantidad in resultado:
    print(f'{estado:7} | {cantidad:18}')

print('')

consulta_2 = '''
SELECT St.state as estado, ROUND(AVG(S.sellingprice), 2) as precio_promedio
FROM Sales S
JOIN Vehicles V ON S.vin = V.vin
JOIN States St ON V.state_id = St.state_id
GROUP BY St.state_id
ORDER BY precio_promedio DESC
LIMIT 5
'''

resultado_2 = ejecutar(consulta_2)

print(' Estado | Precio promedio') 
print('-------------------------')
for estado, precio_promedio in resultado_2:
    print(f'{estado:7} | {precio_promedio:15}')

 Estado | Cantidad de ventas
----------------------------
ca      |               9641
fl      |               7748
tx      |               4703
ga      |               3736
pa      |               3016

 Estado | Precio promedio
-------------------------
tn      |        17089.83
ms      |        16166.67
nv      |        15532.91
co      |        15427.14
il      |        14704.32


## 6. Encuentre el fabricante con el mayor ingreso total por ventas

In [76]:
consulta = '''
SELECT Mf.make as fabricante, SUM(S.sellingprice) as ingreso_total
FROM Manufacturers Mf
JOIN Models Md ON Mf.manufacturer_id = Md.manufacturer_id
JOIN Vehicles V ON Md.model_id = V.model_id
JOIN Sales S ON V.vin = S.vin
GROUP BY Mf.manufacturer_id
ORDER BY ingreso_total DESC
LIMIT 1
'''

resultado = ejecutar(consulta)

print('Fabricante con mayor ingreso total por ventas')
print('---------------------------------------------')
for fabricante, ingreso_total in resultado:
    print(f'{fabricante:14} | {ingreso_total:20}')

Fabricante con mayor ingreso total por ventas
---------------------------------------------
Ford           |          139887080.0


# C7E2

In [78]:
connection = sqlite3.connect('football_results.db')
cursor = connection.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tablas:\n")
for table in cursor.fetchall():
    print(table[0])
    cursor.execute(f'PRAGMA table_info([{table[0]}])')
    print(cursor.fetchall())
    print()

Tablas:

Equipos
[(0, 'eid', 'INTEGER', 0, None, 1), (1, 'nombre', 'TEXT', 0, None, 0)]

Temporadas
[(0, 'tid', 'INTEGER', 0, None, 1), (1, 'ano', 'INTEGER', 0, None, 0)]

Ligas
[(0, 'lid', 'INTEGER', 0, None, 1), (1, 'nombre', 'TEXT', 0, None, 0)]

Partidos
[(0, 'pid', 'INTEGER', 0, None, 1), (1, 'liga', 'INTEGER', 0, None, 0), (2, 'temporada', 'INTEGER', 0, None, 0), (3, 'fecha', 'DATE', 0, None, 0), (4, 'equipo', 'INTEGER', 0, None, 0), (5, 'localidad', 'TEXT', 0, None, 0), (6, 'resultado', 'TEXT', 0, None, 0), (7, 'puntos', 'INTEGER', 0, None, 0), (8, 'goles', 'INTEGER', 0, None, 0), (9, 'goles_perdidos', 'INTEGER', 0, None, 0), (10, 'pases_profundos', 'INTEGER', 0, None, 0), (11, 'pases_profundos_permitidos', 'INTEGER', 0, None, 0), (12, 'ppda', 'REAL', 0, None, 0), (13, 'oppda', 'REAL', 0, None, 0)]



## 1. Encuentre para cada liga, la cantidad de equipos distintos que han participado en ella.

In [ ]:
consulta = '''
SELECT L.nombre as liga, COUNT(DISTINCT P.equipo) as cantidad_equipos
FROM Ligas L
JOIN Partidos P ON L.lid = P.liga
GROUP BY P.liga
'''

resultado = ejecutar(consulta)

print('   Liga    | Cantidad de equipos')
print('--------------------------------')
for liga, cantidad_equipos in resultado:
    print(f'{liga:10} | {cantidad_equipos:18}')

   Liga    | Cantidad de equipos
--------------------------------
Bundesliga |                 24
EPL        |                 30
La_liga    |                 30
Ligue_1    |                 29
RFPL       |                 25
Serie_A    |                 30


## 2. Encuentre el nombre del equipo que tiene mejor diferencia entre goles anotados y goles perdidos a lo largo de los años

In [86]:
consulta = '''
SELECT E.nombre as equipo, SUM(P.goles) as goles, SUM(P.goles_perdidos) as goles_perdidos, SUM(P.goles) - SUM(P.goles_perdidos) as diferencia
FROM Equipos E
JOIN Partidos P ON E.eid = P.equipo 
GROUP BY P.equipo 
ORDER BY diferencia DESC 
LIMIT 1
'''

resultado = ejecutar(consulta)

print('Equipo con mayor diferencia de goles')
print('-------------------------------------')
for equipo, goles, goles_perdidos, diferencia in resultado:
    print(f'{equipo:10} | {diferencia:4}')

Equipo con mayor diferencia de goles
-------------------------------------
Barcelona  |  423


## 3. Para cada liga, encuentre la temporada donde hubo mayor diferencia de puntaje entre el primer y último lugar, e indique esta diferencia

In [91]:
consulta = '''
SELECT L.nombre, DPUL.temporada, MAX(DPUL.diferencia) 
FROM Ligas L
JOIN (SELECT PETL.liga, PETL.temporada, MAX(PETL.puntos_totales) - MIN(PETL.puntos_totales) AS diferencia 
    FROM (SELECT P.liga, P.temporada, P.equipo, SUM(P.puntos) AS puntos_totales 
        FROM Partidos P 
        GROUP BY P.liga, P.temporada, P.equipo) 
        AS PETL 
        GROUP BY PETL.liga, PETL.temporada) 
     AS DPUL ON L.lid = DPUL.liga 
GROUP BY DPUL.liga
'''

# Acá se realizaron consultas anidadas, estas son consultas que se realizan dentro de otras consultas.

resultado = ejecutar(consulta)

print('   Liga    | Temporada | Diferencia de puntos')
print('---------------------------------------------')
for liga, temporada, diferencia in resultado:
    print(f'{liga:10} | {temporada:9} | {diferencia:20}')

   Liga    | Temporada | Diferencia de puntos
---------------------------------------------
Bundesliga |         2 |                   63
EPL        |         5 |                   82
La_liga    |         1 |                   74
Ligue_1    |         2 |                   78
RFPL       |         3 |                   55
Serie_A    |         3 |                   76


In [93]:
# Vemos lo que se hubiera obtenido con la primera consulta anidada

consulta = '''
SELECT P.liga, P.temporada, P.equipo, SUM(P.puntos) AS puntos_totales 
FROM Partidos P 
GROUP BY P.liga, P.temporada, P.equipo
'''

resultado = ejecutar(consulta)

print('   Liga    | Temporada | Equipo | Puntos totales')
print('------------------------------------------------')
for liga, temporada, equipo, puntos_totales in resultado:
    print(f'{liga:10} | {temporada:9} | {equipo:6} | {puntos_totales:13}')

   Liga    | Temporada | Equipo | Puntos totales
------------------------------------------------
         1 |         1 |      1 |            79
         1 |         1 |      2 |            35
         1 |         1 |      3 |            61
         1 |         1 |      4 |            44
         1 |         1 |      5 |            49
         1 |         1 |      6 |            35
         1 |         1 |      7 |            43
         1 |         1 |      8 |            48
         1 |         1 |      9 |            40
         1 |         1 |     10 |            37
         1 |         1 |     11 |            46
         1 |         1 |     12 |            66
         1 |         1 |     13 |            69
         1 |         1 |     14 |            43
         1 |         1 |     15 |            36
         1 |         1 |     16 |            40
         1 |         1 |     17 |            34
         1 |         1 |     18 |            31
         1 |         2 |      1 |     

In [94]:
# Vemos lo que se hubiera obtenido con la segunda consulta anidada

consulta = '''
SELECT PETL.liga, PETL.temporada, MAX(PETL.puntos_totales) - MIN(PETL.puntos_totales) AS diferencia
FROM (SELECT P.liga, P.temporada, P.equipo, SUM(P.puntos) AS puntos_totales 
    FROM Partidos P 
    GROUP BY P.liga, P.temporada, P.equipo) 
    AS PETL
GROUP BY PETL.liga, PETL.temporada
'''

resultado = ejecutar(consulta)

print('   Liga    | Temporada | Diferencia de puntos')
print('---------------------------------------------')
for liga, temporada, diferencia in resultado:
    print(f'{liga:10} | {temporada:9} | {diferencia:20}')

   Liga    | Temporada | Diferencia de puntos
---------------------------------------------
         1 |         1 |                   48
         1 |         2 |                   63
         1 |         3 |                   57
         1 |         4 |                   62
         1 |         5 |                   59
         1 |         6 |                   62
         2 |         1 |                   57
         2 |         2 |                   64
         2 |         3 |                   69
         2 |         4 |                   69
         2 |         5 |                   82
         2 |         6 |                   78
         3 |         1 |                   74
         3 |         2 |                   59
         3 |         3 |                   73
         3 |         4 |                   73
         3 |         5 |                   55
         3 |         6 |                   62
         4 |         1 |                   54
         4 |         2 |          

# Basketball

In [97]:
connection = sqlite3.connect('nba.sqlite')
cursor = connection.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tablas:\n")
for table in cursor.fetchall():
    print(table[0])
    cursor.execute(f'PRAGMA table_info([{table[0]}])')
    print(cursor.fetchall())
    print()

Tablas:

Salaries
[(0, 'Player', 'TEXT', 0, None, 0), (1, 'Team', 'TEXT', 0, None, 0), (2, 'Salary', 'REAL', 0, None, 0)]

Stats
[(0, 'Year', 'REAL', 0, None, 0), (1, 'Player', 'TEXT', 0, None, 0), (2, 'Position', 'TEXT', 0, None, 0), (3, 'Age', 'REAL', 0, None, 0), (4, 'Team', 'TEXT', 0, None, 0), (5, 'Games', 'REAL', 0, None, 0), (6, 'GamesStarted', 'INTEGER', 0, None, 0), (7, 'MinutesPlayed', 'REAL', 0, None, 0), (8, '3Points', 'INTEGER', 0, None, 0), (9, '3PAttempts', 'INTEGER', 0, None, 0), (10, '2Points', 'REAL', 0, None, 0), (11, '2PAttempts', 'REAL', 0, None, 0), (12, 'FreeThrows', 'REAL', 0, None, 0), (13, 'FTAttempts', 'REAL', 0, None, 0), (14, 'Fouls', 'REAL', 0, None, 0), (15, 'Points', 'REAL', 0, None, 0)]



## 1. Complete los valores faltantes para la columna 3Points, cuando sea posible inferirlos a partir del resto de las columnas.

In [101]:
consulta = '''
SELECT "3Points", "2Points", FreeThrows, Points 
FROM Stats S 
WHERE "3Points" IS NULL
'''

resultado = ejecutar(consulta)

print('  3 puntos |  2 puntos | Tiros libres | Puntos')
print('-----------------------------------------------')
for tres_puntos, dos_puntos, tiros_libres, puntos in resultado:
    print(f'{tres_puntos} | {dos_puntos} | {tiros_libres} | {puntos}')

  3 puntos |  2 puntos | Tiros libres | Puntos
-----------------------------------------------
None | 40.0 | 44.0 | 406.0
None | 33.0 | 45.0 | 222.0
None | 29.0 | 43.0 | 209.0
None | 123.0 | 83.0 | 515.0
None | 477.0 | 220.0 | 1243.0
None | 259.0 | 80.0 | 643.0
None | 113.0 | 96.0 | 532.0
None | 16.0 | 12.0 | 86.0
None | 128.0 | 102.0 | 532.0
None | 74.0 | 70.0 | 329.0
None | 54.0 | 32.0 | 203.0
None | 78.0 | 45.0 | 246.0
None | 119.0 | 129.0 | 979.0
None | 607.0 | 471.0 | 1832.0
None | 451.0 | 304.0 | 1659.0
None | 135.0 | 93.0 | 936.0
None | 42.0 | 19.0 | 262.0
None | 100.0 | 131.0 | 616.0
None | 26.0 | 11.0 | 324.0
None | 59.0 | 28.0 | 215.0
None | 33.0 | 31.0 | 106.0
None | 137.0 | 40.0 | 419.0
None | 89.0 | 44.0 | 381.0
None | 521.0 | 242.0 | 1518.0
None | 98.0 | 70.0 | 527.0
None | 78.0 | 50.0 | 413.0
None | 20.0 | 20.0 | 114.0
None | 208.0 | 143.0 | 820.0
None | 258.0 | 243.0 | 1164.0
None | 9.0 | 9.0 | 33.0
None | 203.0 | 119.0 | 801.0
None | 414.0 | 282.0 | 1779.0
None | 24.0 

Se puede ver que hay varias columnas que tienen el dato como *None*

In [102]:
consulta = '''
UPDATE Stats 
SET "3Points" = (Points - ((2*"2Points") + FreeThrows))/3 
WHERE "3Points" IS NULL 
AND "2Points" IS NOT NULL 
AND FreeThrows IS NOT NULL 
AND Points IS NOT NULL
'''

cursor.execute(consulta)

In [103]:
# Vemos si siguen quedando valores nulos en la columna "3Points"

consulta = '''
SELECT "3Points", "2Points", FreeThrows, Points
FROM Stats S
WHERE "3Points" IS NULL
'''

resultado = ejecutar(consulta)

print('  3 puntos |  2 puntos | Tiros libres | Puntos')
print('-----------------------------------------------')
for tres_puntos, dos_puntos, tiros_libres, puntos in resultado:
    print(f'{tres_puntos} | {dos_puntos} | {tiros_libres} | {puntos}')

  3 puntos |  2 puntos | Tiros libres | Puntos
-----------------------------------------------


No hay

## 2. Encuentre los 5 equipos de la NBA para los que sus 5 jugadores más veteranos cometieron más fouls. Para cada equipo, imprima además la cantidad de fouls agregada que realizaron estos jugadores. Ordene los resultados de manera ascendente en base a los fouls

In [109]:
consulta = '''
SELECT S.Team, SUM(S.Fouls) as TotalEquipo
FROM Stats S
WHERE S.Player IN (
    SELECT S2.Player
    FROM Stats S2
    WHERE S2.Team = S.Team
    ORDER BY S2.Age DESC
    LIMIT 5
)
GROUP BY S.Team
ORDER BY TotalEquipo DESC
LIMIT 5
'''

resultado = ejecutar(consulta)

print('Equipo | Total de faltas')
print('------------------------')
for equipo, total_equipo in resultado:
    print(f'{equipo:6} | {total_equipo:15}')


Equipo | Total de faltas
------------------------
MEM    |           698.0
BOS    |           686.0
POR    |           661.0
NYK    |           588.0
TOR    |           576.0


## 2. Encuentre los 10 jugadores más costosos de la NBA de acuerdo a la cantidad de dinero que les pagaron por cada punto que convirtieron. Para cada jugador, imprima además su sueldo total en la temporada. Ordene los resultados de manera descendente, de acuerdo al sueldo total.

In [118]:
consulta = '''
SELECT jugador, salario_total, salario_punto 
FROM (
    SELECT S.Player as jugador, SUM(A.Salary) as salario_total, ROUND(SUM(A.Salary)/SUM(S.Points), 0) as salario_punto 
    FROM Stats S
    JOIN Salaries A
    ON S.Player = A.Player AND S.Team = A.Team
    GROUP BY S.Player 
    ORDER BY salario_punto DESC 
    LIMIT 10
    ) 
ORDER BY salario_total DESC
'''

resultado = ejecutar(consulta)

print('     Jugador     | Salario total | Salario por punto')
print('----------------------------------------------------')
for jugador, salario_total, salario_punto in resultado:
    print(f'{jugador:17} | {salario_total:13} | {salario_punto:16}')

     Jugador     | Salario total | Salario por punto
----------------------------------------------------
Chandler Parsons  |    23112004.0 |         110057.0
Joakim Noah       |    17765000.0 |          76573.0
Ian Mahinmi       |    16661641.0 |          96310.0
Jerryd Bayless    |     9000000.0 |         272727.0
Lance Stephenson  |     4180000.0 |          97209.0
Udonis Haslem     |     2328652.0 |          75118.0
Chris McCullough  |     1471382.0 |        1471382.0
Josh Huestis      |     1471382.0 |         105099.0
Chinanu Onuaku    |     1312611.0 |          93758.0
Michael Gbinije   |      500000.0 |         125000.0


## 3. Para los jugadores que fueron parte de 2 o más equipos distintos durante la temporada, encuentre aquel que tuvo la mejora más significativa en cuanto a puntos convertidos por partido.

In [123]:
# P1 ve los jugadores que fueron parte de más de un equipo, mientras que P2 calcula el
# promedio de puntos por juego de cada jugador.

consulta= ''' 
SELECT Jugadores.Player, Jugadores.Mejora
FROM (
    SELECT P2.Player as Player, ROUND((MAX(P2.PpG) - MIN (P2.PpG)),2) as Mejora
    FROM (
        SELECT s.Player as Player
        FROM Stats s
        GROUP BY s.Player
        HAVING COUNT(*) > 1
    ) AS P1, (
        SELECT s2.Player as Player, (s2.Points / s2.Games) as PpG
        FROM Stats s2
    ) AS P2
    WHERE P1.Player = P2.Player
    GROUP BY P2.Player
) as Jugadores
ORDER BY Jugadores.Mejora DESC
LIMIT 1
'''

resultado = ejecutar(consulta)

print('     Jugador     | Mejora')
print('-------------------------')
for jugador, mejora in resultado:
    print(f'{jugador:16} | {mejora:6}')

     Jugador     | Mejora
-------------------------
Terrence Jones   |  11.45
